# Land Cover Analysis & Visualization (Nigeria) - ESA Dynamic World (2020-2024)

This notebook analyzes land cover changes in Nigeria using ESA Dynamic World data. 
It includes:
- **Modular Analysis Class**: `LandCoverAnalyzer` for streamlined processing.
- **Interactive Visualizations**: Altair charts for exploring area changes.
- **Change Detection**: Visualizing areas that have changed class between years.
- **Transition Matrix**: Quantifying land cover transitions (e.g., Forest to Shrubland).

In [1]:
## Install packages
# Note: 'ee' in pip is not Earth Engine. We need 'earthengine-api'.
!pip install geemap earthengine-api rasterio geopandas matplotlib numpy seaborn pandas folium ipyleaflet altair

In [2]:
# Import Packages
import ee
import geemap
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import ipywidgets as widgets
from ipywidgets import HBox, VBox

# Initialize Earth Engine
try:
    ee.Initialize()
    print("EE initialized!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize()
    print("EE initialized after authentication!")


Successfully saved authorization token.
EE initialized after authentication!


In [3]:
# ---------------------------------------------------------------------------------------
# Configuration & Constants
# ---------------------------------------------------------------------------------------

DW_CLASSES = {
    0: 'Water',
    1: 'Trees',
    2: 'Grass',
    3: 'Flooded vegetation',
    4: 'Crops',
    5: 'Shrub & Scrub',
    6: 'Built Area',
    7: 'Bare Ground',
    8: 'Snow & Ice',
}

PALETTE = [
    '#419BDF',  # 0 Water
    '#397D49',  # 1 Trees
    '#88B053',  # 2 Grass
    '#7A87C6',  # 3 Flooded vegetation
    '#E49635',  # 4 Crops
    '#DFC35A',  # 5 Shrub & Scrub
    '#C4281B',  # 6 Built Area
    '#A59B8F',  # 7 Bare Ground
    '#B39FE1',  # 8 Snow & Ice
]

LEGEND_DICT = {DW_CLASSES[i]: PALETTE[i] for i in range(9)}

YEARS = list(range(2020, 2025))
ZOOM_LEVEL = 6
SCALE_COUNTRY = 100  # Adjusted for speed, use 30 for higher precision
TILESCALE = 4

# Define Area of Interest (AOI)
gaul_countries = ee.FeatureCollection('FAO/GAUL/2015/level0')
nigeria_fc = gaul_countries.filter(ee.Filter.eq('ADM0_NAME', 'Nigeria'))
AOI = nigeria_fc.geometry()
AOI_SIMP = AOI.simplify(200) # Simplified geometry for faster stats

In [4]:
# ---------------------------------------------------------------------------------------
# LandCoverAnalyzer Class
# ---------------------------------------------------------------------------------------

class LandCoverAnalyzer:
    def __init__(self, aoi, years, classes, palette):
        self.aoi = aoi
        self.years = years
        self.classes = classes
        self.palette = palette
        self.dw_collection = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        self.annual_images = self._prepare_annual_images()

    def _prepare_annual_images(self):
        """Prepares annual composite images (mode of monthly labels)."""
        annual = {}
        for year in self.years:
            ic = (self.dw_collection.filterBounds(self.aoi)
                    .filter(ee.Filter.calendarRange(year, year, 'year'))
                    .select('label'))
            # Reduce to mode -> band 'label_mode', then rename to 'label'
            img = ic.reduce(ee.Reducer.mode()).select('label_mode').rename('label')
            annual[year] = img
        return annual

    def get_image(self, year):
        """Returns the annual image for a specific year."""
        return self.annual_images.get(year)

    def calculate_area_stats(self, scale=SCALE_COUNTRY, tilescale=TILESCALE):
        """Calculates area statistics for all years."""
        all_stats = []
        
        for year in self.years:
            img = self.get_image(year)
            # Ensure 'label' band
            lbl = img.select('label').clip(self.aoi)
            area_img = ee.Image.pixelArea().rename('area')
            pair = area_img.addBands(lbl.int())

            reduced = pair.reduceRegion(
                reducer=ee.Reducer.sum().group(groupField=1, groupName='label'),
                geometry=self.aoi,
                scale=scale,
                maxPixels=1e13,
                tileScale=tilescale
            )

            groups = ee.Dictionary(reduced).get('groups')
            groups_list = ee.List(groups).getInfo() if groups is not None else []
            
            # Process results
            by_code_ha = {int(g['label']): g['sum'] / 10000.0 for g in groups_list if g}
            
            for code in range(9):
                class_name = self.classes[code]
                hectares = by_code_ha.get(code, 0.0)
                all_stats.append({'Year': str(year), 'Class': class_name, 'Hectares': hectares})
                
        return pd.DataFrame(all_stats)

    def calculate_transition_matrix(self, start_year, end_year, scale=SCALE_COUNTRY, tilescale=TILESCALE):
        """Calculates the transition matrix between two years."""
        img1 = self.get_image(start_year).select('label').rename('start')
        img2 = self.get_image(end_year).select('label').rename('end')
        
        # Combine images: start * 10 + end (e.g., class 1 to 2 becomes 12)
        # This assumes classes 0-8. Max value 88.
        transition = img1.multiply(10).add(img2).rename('trans')
        
        area_img = ee.Image.pixelArea().rename('area')
        pair = area_img.addBands(transition.int())
        
        reduced = pair.reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName='trans'),
            geometry=self.aoi,
            scale=scale,
            maxPixels=1e13,
            tileScale=tilescale
        )
        
        groups = ee.Dictionary(reduced).get('groups')
        groups_list = ee.List(groups).getInfo() if groups is not None else []
        
        matrix_data = []
        for g in groups_list:
            trans_code = int(g['trans'])
            start_code = trans_code // 10
            end_code = trans_code % 10
            area_ha = g['sum'] / 10000.0
            
            matrix_data.append({
                'Start_Class': self.classes.get(start_code, f'Unknown {start_code}'),
                'End_Class': self.classes.get(end_code, f'Unknown {end_code}'),
                'Hectares': area_ha
            })
            
        return pd.DataFrame(matrix_data)

    def get_change_mask(self, start_year, end_year):
        """Returns an image representing changed pixels."""
        img1 = self.get_image(start_year).select('label')
        img2 = self.get_image(end_year).select('label')
        # Change is where img1 != img2
        changed = img1.neq(img2).selfMask()
        return changed

    def explain_transitions(self, df_trans, top_n=5):
        """Explains the transition matrix results with robust insights."""
        total_area = df_trans['Hectares'].sum()
        
        # Filter for actual changes (Start != End)
        changes = df_trans[df_trans['Start_Class'] != df_trans['End_Class']].copy()
        
        # --- 1. General Stats ---
        total_changed = changes['Hectares'].sum()
        pct_changed = (total_changed / total_area) * 100 if total_area > 0 else 0
        
        print(f"--- Transition Analysis Report ---")
        print(f"Total Area Analyzed: {total_area:,.0f} ha")
        print(f"Total Area Changed: {total_changed:,.0f} ha ({pct_changed:.1f}%)")
        
        # --- 2. Deforestation Analysis ---
        # Trees -> Crops, Bare Ground, Shrub & Scrub, Built Area
        deforestation_targets = ['Crops', 'Bare Ground', 'Shrub & Scrub', 'Built Area']
        deforestation = changes[
            (changes['Start_Class'] == 'Trees') & 
            (changes['End_Class'].isin(deforestation_targets))
        ]
        total_deforestation = deforestation['Hectares'].sum()
        
        print(f"\n1. Deforestation Insights:")
        print(f"   - Total Forest Loss: {total_deforestation:,.0f} ha")
        if total_deforestation > 0:
            print(f"   - Main Drivers:")
            for _, row in deforestation.sort_values('Hectares', ascending=False).iterrows():
                print(f"     * To {row['End_Class']}: {row['Hectares']:,.0f} ha")

        # --- 3. Urbanization ---
        # Any -> Built Area (excluding Built Area -> Built Area which is filtered out)
        urbanization = changes[changes['End_Class'] == 'Built Area']
        total_urbanization = urbanization['Hectares'].sum()
        print(f"\n2. Urbanization Trends:")
        print(f"   - New Built Area: {total_urbanization:,.0f} ha")
        
        # --- 4. Agricultural Expansion ---
        # Any -> Crops
        ag_expansion = changes[changes['End_Class'] == 'Crops']
        total_ag_expansion = ag_expansion['Hectares'].sum()
        print(f"\n3. Agricultural Expansion:")
        print(f"   - New Crop Area: {total_ag_expansion:,.0f} ha")
        
        # --- 5. Regrowth/Afforestation ---
        # Any -> Trees
        regrowth = changes[changes['End_Class'] == 'Trees']
        total_regrowth = regrowth['Hectares'].sum()
        print(f"\n4. Forest Regrowth:")
        print(f"   - New Tree Cover: {total_regrowth:,.0f} ha")
        
        # --- 6. Top Major Transitions ---
        print(f"\n5. Top {top_n} Specific Transitions:")
        changes_sorted = changes.sort_values(by='Hectares', ascending=False).reset_index(drop=True)
        for i, row in changes_sorted.head(top_n).iterrows():
            print(f"   {i+1}. {row['Start_Class']} -> {row['End_Class']}: {row['Hectares']:,.0f} ha")

In [5]:
# Initialize Analyzer
analyzer = LandCoverAnalyzer(AOI_SIMP, YEARS, DW_CLASSES, PALETTE)

### Interactive Area Charts

In [6]:
# Calculate Stats
print("Calculating area statistics... this may take a moment.")
df_stats = analyzer.calculate_area_stats(scale=SCALE_COUNTRY)

# Interactive Chart with Altair
chart = alt.Chart(df_stats).mark_bar().encode(
    x=alt.X('Year:O', axis=alt.Axis(title='Year')),
    y=alt.Y('Hectares:Q', axis=alt.Axis(title='Area (Hectares)')),
    color=alt.Color('Class:N', scale=alt.Scale(domain=list(DW_CLASSES.values()), range=PALETTE), legend=alt.Legend(title="Land Cover Class")),
    column=alt.Column('Class:N', header=alt.Header(titleOrient="bottom", labelOrient="bottom")),
    tooltip=['Year', 'Class', alt.Tooltip('Hectares', format=',.0f')]
).properties(
    title='Land Cover Area Trends (2020-2024)',
    width=150
)

chart.display()

Calculating area statistics... this may take a moment.


alt.Chart(...)

### Map Visualization & Change Detection

In [7]:
# Display Maps
def display_maps(start_year, end_year):
    m = geemap.Map()
    m.centerObject(AOI, ZOOM_LEVEL)
    
    # Add Start Year Layer
    img_start = analyzer.get_image(start_year).clip(AOI)
    m.addLayer(img_start, {"min": 0, "max": 8, "palette": PALETTE}, f"Land Cover {start_year}")
    
    # Add End Year Layer
    img_end = analyzer.get_image(end_year).clip(AOI)
    m.addLayer(img_end, {"min": 0, "max": 8, "palette": PALETTE}, f"Land Cover {end_year}")
    
    # Add Change Layer
    change_mask = analyzer.get_change_mask(start_year, end_year).clip(AOI)
    m.addLayer(change_mask, {"palette": ['red']}, f"Changes ({start_year}-{end_year})")
    
    m.add_legend(title="ESA Dynamic World", legend_dict=LEGEND_DICT)
    m.addLayerControl()
    return m

display_maps(2020, 2024)

Map(center=[9.589444610453368, 8.089338153274143], controls=(WidgetControl(options=['position', 'transparent_b…

### Transition Matrix Analysis
This matrix shows the area (in hectares) that transitioned from one class to another between the start and end years.

**Automated Insights**:
The analysis below also provides a detailed report on:
- **Deforestation**: Forest loss to specific classes.
- **Urbanization**: Expansion of built-up areas.
- **Agricultural Expansion**: New crop areas.
- **Regrowth**: New tree cover.

In [8]:
print("Calculating transition matrix... this may take a moment.")
df_trans = analyzer.calculate_transition_matrix(2020, 2024, scale=SCALE_COUNTRY)

# Pivot for readable matrix format
pivot_trans = df_trans.pivot(index='Start_Class', columns='End_Class', values='Hectares').fillna(0)

# Display as a heatmap using Altair
heatmap = alt.Chart(df_trans).mark_rect().encode(
    x=alt.X('End_Class:N', title='To Class (2024)'),
    y=alt.Y('Start_Class:N', title='From Class (2020)'),
    color=alt.Color('Hectares:Q', scale=alt.Scale(scheme='viridis')),
    tooltip=['Start_Class', 'End_Class', alt.Tooltip('Hectares', format=',.0f')]
).properties(
    title='Land Cover Transition Matrix (2020 vs 2024)',
    width=600,
    height=600
)

display(pivot_trans.style.format("{:,.0f}").background_gradient(cmap="viridis", axis=None))
heatmap.display()

# Explain results
analyzer.explain_transitions(df_trans)

Calculating transition matrix... this may take a moment.


End_Class,Bare Ground,Built Area,Crops,Flooded vegetation,Grass,Shrub & Scrub,Snow & Ice,Trees,Water
Start_Class,,,,,,,,,
Bare Ground,"3,771,756","113,784","1,183,524","26,569","34,043","2,844,125",7,"50,677","53,692"
Built Area,"4,608","2,316,575","14,680",333,132,"56,933",5,"92,810","1,792"
Crops,"802,578","73,429","9,639,947","59,114","148,988","4,799,782",0,"751,673","40,121"
Flooded vegetation,"6,497",857,"34,350","354,575","34,288","15,295",27,"104,576","52,761"
Grass,"6,748",117,"54,892","24,409","169,253","35,426",0,"32,339","9,817"
Shrub & Scrub,"946,766","260,683","2,964,141","43,825","95,882","26,277,792",3,"3,722,577","32,863"
Snow & Ice,1,1,1,4,0,2,0,6,1
Trees,"10,994","233,747","588,340","118,627","79,650","3,680,130",61,"22,543,195","37,830"
Water,"35,261","2,917","14,553","228,499","5,191","7,536",137,"174,348","960,538"


alt.Chart(...)

--- Transition Analysis Report ---
Total Area Analyzed: 90,884,008 ha
Total Area Changed: 24,850,377 ha (27.3%)

1. Deforestation Insights:
   - Total Forest Loss: 4,513,211 ha
   - Main Drivers:
     * To Shrub & Scrub: 3,680,130 ha
     * To Crops: 588,340 ha
     * To Built Area: 233,747 ha
     * To Bare Ground: 10,994 ha

2. Urbanization Trends:
   - New Built Area: 685,535 ha

3. Agricultural Expansion:
   - New Crop Area: 4,854,482 ha

4. Forest Regrowth:
   - New Tree Cover: 4,929,006 ha

5. Top 5 Specific Transitions:
   1. Crops -> Shrub & Scrub: 4,799,782 ha
   2. Shrub & Scrub -> Trees: 3,722,577 ha
   3. Trees -> Shrub & Scrub: 3,680,130 ha
   4. Shrub & Scrub -> Crops: 2,964,141 ha
   5. Bare Ground -> Shrub & Scrub: 2,844,125 ha
